LongWanjuan: Towards Systematic Measurement for Long Text Quality

In [4]:
import nltk
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

C:\Users\Makai\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
nltk.download('punkt')

# Load pre-trained language model
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Makai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
C:\Users\Makai\AppData\Roaming\Python\Python312\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [36]:
# Function to measure coherence
def coherence_metrics(text, window_size=4096):
    # Tokenize text
    tokens = tokenizer("\n" + text + "\n", return_tensors='pt', truncation=True, max_length=window_size)
    input_ids = tokens.input_ids
    
    # Split the text into smaller contexts (short and long contexts)
    short_context = input_ids[:, :window_size // 2]
    long_context = input_ids
    
    # Predict on short and long contexts
    with torch.no_grad():
        outputs_short = model(short_context, labels=short_context)
        outputs_long = model(long_context, labels=long_context)
    
    # Calculate accuracy and log-likelihood difference
    coherence_acc_short = (outputs_short.logits.argmax(-1) == short_context).float().mean().item()
    coherence_acc_long = (outputs_long.logits.argmax(-1) == long_context).float().mean().item()
    
    coherence_diff = (coherence_acc_long - coherence_acc_short) / coherence_acc_long
    return coherence_acc_long, coherence_acc_short, coherence_diff

# Function to measure cohesion (connectives and pronouns)
def cohesion_metrics(text):
    connectives = ['but', 'however', 'therefore', 'yet', 'since', 'because', 'although']  # Add more connectives
    pronouns = ['he', 'she', 'it', 'they', 'we', 'I', 'you']  # Add more pronouns

    sentences = nltk.sent_tokenize(text)
    num_connectives = sum(1 for word in nltk.word_tokenize(text.lower()) if word in connectives)
    num_pronouns = sum(1 for word in nltk.word_tokenize(text.lower()) if word in pronouns)
    
    cohesion_conn = num_connectives / len(sentences)
    cohesion_pron = num_pronouns / len(sentences)
    
    return cohesion_conn, cohesion_pron

# Function to measure complexity (TTR and average paragraph length)
def complexity_metrics(text):
    words = nltk.word_tokenize(text)
    unique_words = set(words)
    ttr = len(unique_words) / len(words)
    
    paragraphs = text.split('\n\n')
    avg_para_length = sum(len(nltk.word_tokenize(para)) for para in paragraphs) / len(paragraphs)
    
    return ttr, avg_para_length

Coherence - Long Acc: , Short Acc: , Diff: 
-----------------------------------------
- **Long Acc**: This value represents the accuracy of predicting the next word or token using a longer context window from the text. Higher values indicate better predictive performance over extended portions of the text, showing how well the text flows and maintains consistency over long distances.
  
- **Short Acc**: This value represents the accuracy of predicting the next word or token using a shorter context window from the text. Lower values compared to long acc would indicate that using more of the previous text improves the prediction accuracy, implying better coherence.

- **Diff**: This is the difference in accuracy between the long and short contexts, calculated as `(Long Acc - Short Acc) / Long Acc`. A higher value suggests a significant improvement in predictive ability when more context is provided, indicating strong coherence across the text. A smaller value implies that the short context is nearly as effective as the long one, indicating weaker coherence.

Cohesion - Connectives: , Pronouns:
-----------------------------------
- **Connectives**: This value measures the density of connective words (e.g., "but", "therefore") within the text. A higher value indicates strong logical connections between sentences or sections, showing good flow and structure. A low value may imply disjointed or weakly connected ideas in the text.

- **Pronouns**: This value measures the density of pronouns (e.g., "he", "it", "they") within the text. A higher value indicates frequent reference to previously mentioned entities, which helps in maintaining cohesion. If this value is too low, the text might lack connections between different parts of the discourse, making it harder to follow.

Complexity - TTR: , Avg Para Length: 
-----------------------------------
- **TTR (Type-Token Ratio)**: This measures the lexical richness or diversity of the text. A higher value indicates that the text uses a wider variety of words, which typically reflects more sophisticated or varied language. A lower value suggests repetition or simpler language usage.

- **Avg Para Length**: This value represents the average number of words per paragraph. Longer paragraphs may indicate more complex sentence structures and a denser flow of ideas, while shorter paragraphs could reflect simpler or more fragmented writing.


In [17]:
text_prog = [
"""
Bees are essential pollinators that play a crucial role in maintaining biodiversity and supporting ecosystems. These industrious insects are responsible for pollinating many of the plants that produce fruits, vegetables, and nuts, contributing to about one-third of the food we consume. Honeybees, in particular, are known for their ability to produce honey, but their most significant contribution is through pollination, which supports both wild plant species and agricultural crops. Bees live in highly organized colonies, with a social structure that includes worker bees, drones, and a single queen. Unfortunately, bee populations are declining due to factors like habitat loss, pesticide use, climate change, and disease, which raises concerns for global food security and environmental health.
""", 
"""
Our ecosystem relies heavily on the industrious bee, a pollination powerhouse. These buzzing insects are the unsung heroes behind a vast array of plants we rely on, especially those providing fruits, vegetables, and nuts that grace our tables daily. Honeybees, masters of honey production, stand out as particularly crucial pollinators. Within their structured colonies, a complex social hierarchy thrives. Alarmingly, bee populations are facing a steep decline. Habitat loss, pesticide use, a changing climate, and the spread of diseases all contribute to this worrying trend. Such a decline raises serious concerns for the future of global food security and the well-being of our environment.
""", 
"""
The diligent bee, a champion pollinator, plays a critical role in our ecosystem. These buzzing wonders are the unheralded force behind countless plants essential to our survival, especially those yielding the fruits, vegetables, and nuts that grace our tables. Honeybees, masters of their craft, are particularly vital pollinators. Their intricate colonies buzz with a complex social structure. However, bee populations are alarmingly dwindling. Habitat loss, pesticide use, climate change, and disease spread all contribute to this worrisome pattern. This decline casts a shadow on the future of global food security and the well-being of our environment.
"""
]

In [41]:
for sample in text_prog:
    print(f"\n\nSample: {sample}")

    # Coherence
    coh_acc_long, coh_acc_short, coh_diff = coherence_metrics(sample)
    print(f"Coherence - Long Acc: {coh_acc_long}, Short Acc: {coh_acc_short}, Diff: {coh_diff}")

    # Cohesion
    cohesion_conn, cohesion_pron = cohesion_metrics(sample)
    print(f"Cohesion - Connectives: {cohesion_conn}, Pronouns: {cohesion_pron}")

    # Complexity
    ttr, avg_para_len = complexity_metrics(sample)
    print(f"Complexity - TTR: {ttr}, Avg Para Length: {avg_para_len}")



Sample: 
Bees are essential pollinators that play a crucial role in maintaining biodiversity and supporting ecosystems. These industrious insects are responsible for pollinating many of the plants that produce fruits, vegetables, and nuts, contributing to about one-third of the food we consume. Honeybees, in particular, are known for their ability to produce honey, but their most significant contribution is through pollination, which supports both wild plant species and agricultural crops. Bees live in highly organized colonies, with a social structure that includes worker bees, drones, and a single queen. Unfortunately, bee populations are declining due to factors like habitat loss, pesticide use, climate change, and disease, which raises concerns for global food security and environmental health.

Coherence - Long Acc: 0.006849315017461777, Short Acc: 0.006849315017461777, Diff: 0.0
Cohesion - Connectives: 0.2, Pronouns: 0.2
Complexity - TTR: 0.6814814814814815, Avg Para Length: 13

In [37]:
# Direct string
test = """
Bees are essential pollinators that play a crucial role in maintaining biodiversity and supporting ecosystems. These industrious insects are responsible for pollinating many of the plants that produce fruits, vegetables, and nuts, contributing to about one-third of the food we consume. Honeybees, in particular, are known for their ability to produce honey, but their most significant contribution is through pollination, which supports both wild plant species and agricultural crops. Bees live in highly organized colonies, with a social structure that includes worker bees, drones, and a single queen. Unfortunately, bee populations are declining due to factors like habitat loss, pesticide use, climate change, and disease, which raises concerns for global food security and environmental health.
"""
coh_acc_long, coh_acc_short, coh_diff = coherence_metrics(test)
print(f"Coherence - Long Acc: {coh_acc_long}, Short Acc: {coh_acc_short}, Diff: {coh_diff}")


Coherence - Long Acc: 0.006849315017461777, Short Acc: 0.006849315017461777, Diff: 0.0


---

In [28]:
# Example usage
text = """
Bees are remarkable insects that play a crucial role in our ecosystem, primarily through pollination. Transferring pollen between flowers, they help plants reproduce, supporting biodiversity and the growth of crops that humans and animals rely on for food. Honeybees, in particular, are known for producing honey and beeswax, both valuable human resources. There are over 20,000 species of bees, ranging from solitary bees to highly social species like the honeybee.
"""

# Coherence
coh_acc_long, coh_acc_short, coh_diff = coherence_metrics(text)
print(f"Coherence - Long Acc: {coh_acc_long}, Short Acc: {coh_acc_short}, Diff: {coh_diff}")

# Cohesion
cohesion_conn, cohesion_pron = cohesion_metrics(text)
print(f"Cohesion - Connectives: {cohesion_conn}, Pronouns: {cohesion_pron}")

# Complexity
ttr, avg_para_len = complexity_metrics(text)
print(f"Complexity - TTR: {ttr}, Avg Para Length: {avg_para_len}")

Coherence - Long Acc: 0.021739130839705467, Short Acc: 0.021739130839705467, Diff: 0.0
Cohesion - Connectives: 0.0, Pronouns: 0.25
Complexity - TTR: 0.7530864197530864, Avg Para Length: 81.0


In [42]:
# Example usage
text = """
Bees are insects that have an impact, in our environment by mainly aiding in pollination activities which support plant reproduction and the diversity of crops crucial for our food sources and animal sustenance alike. Honeybees stand out for their production of honey and beeswax that hold value for humans as resources within our society. The bee population comprises than 20 thousand species that include bees as well as highly sociable species such, as the honeybee.
"""

# Coherence
coh_acc_long, coh_acc_short, coh_diff = coherence_metrics(text)
print(f"Coherence - Long Acc: {coh_acc_long}, Short Acc: {coh_acc_short}, Diff: {coh_diff}")

# Cohesion
cohesion_conn, cohesion_pron = cohesion_metrics(text)
print(f"Cohesion - Connectives: {cohesion_conn}, Pronouns: {cohesion_pron}")

# Complexity
ttr, avg_para_len = complexity_metrics(text)
print(f"Complexity - TTR: {ttr}, Avg Para Length: {avg_para_len}")

Coherence - Long Acc: 0.010989011265337467, Short Acc: 0.010989011265337467, Diff: 0.0
Cohesion - Connectives: 0.0, Pronouns: 0.0
Complexity - TTR: 0.775, Avg Para Length: 80.0
